In [1]:
import os
import math
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import soundfile as sf
import joblib

# Force timm into offline mode BEFORE importing it
# Prevents HuggingFace Hub network lookup hang on Kaggle internet-off env
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TIMM_FUSED_ATTN'] = '0'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import timm

class CFG:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
    SAMPLE_SUB_CSV = os.path.join(ROOT_DIR, 'sample_submission.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'test_soundscapes')

    # Update these paths to wherever you saved your trained model in Kaggle Datasets
    MODEL_PATH = '/kaggle/input/models/punyakdei/train-geo-alter2/pytorch/default/1/best_model_vit_fold_0.pth'
    KNN_PATH   = '/kaggle/input/models/punyakdei/train-geo-alter2/pytorch/default/1/knn_spatial_fold_0.pkl'

    SR               = 32000
    WINDOW_SECONDS   = 5.0
    HOP_SECONDS      = 2.5
    CHUNK_LENGTH     = int(SR * WINDOW_SECONDS)
    HOP_LENGTH_AUDIO = int(SR * HOP_SECONDS)

    BACKBONE_NAME        = 'vit_base_patch16_224'
    IMAGE_SIZE           = (224, 224)
    CONFIDENCE_THRESHOLD = 0.85
    KNN_ALPHA            = 0.3
    SUB_BATCH_SIZE       = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [2]:
class DualHeadViT(nn.Module):
    def __init__(self, backbone_name=CFG.BACKBONE_NAME, num_classes=234):
        super().__init__()
        # num_classes=0 tells timm to output raw features (no classifier head)
        # This also avoids any need to call reset_classifier()
        self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0, in_chans=3)
        in_features = self.backbone.num_features  # e.g. 768 for vit_base
        self.acoustic_head = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.2), nn.Linear(512, num_classes)
        )
        self.geo_head = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.2), nn.Linear(512, 2)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.acoustic_head(features), self.geo_head(features)

# Load labels
sample_sub = pd.read_csv(CFG.SAMPLE_SUB_CSV)
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
NUM_CLASSES = len(submission_labels)
print(f"NUM_CLASSES = {NUM_CLASSES}")

# Load model weights - hard assert to catch missing dataset attachment immediately
assert os.path.exists(CFG.MODEL_PATH), (
    f"FATAL: Model weights not found at:\n  {CFG.MODEL_PATH}\n"
    f"Ensure your trained model Kaggle Dataset is attached to this notebook."
)
model = DualHeadViT(num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(CFG.MODEL_PATH, map_location=device))
model.eval()
print(f"Model loaded successfully from {CFG.MODEL_PATH}")

# Load KNN (optional - gracefully skip if missing)
knn_model = None
if os.path.exists(CFG.KNN_PATH):
    knn_model = joblib.load(CFG.KNN_PATH)
    print(f"KNN model loaded from {CFG.KNN_PATH}")
else:
    print(f"WARNING: KNN not found at {CFG.KNN_PATH}. Running acoustic-only mode.")

# Mel transform on GPU
mel_transform = T.MelSpectrogram(
    sample_rate=CFG.SR, n_fft=2048, hop_length=512,
    n_mels=128, f_min=20, f_max=16000
).to(device)

# Pre-allocate ImageNet normalization tensors on GPU once
IMG_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
IMG_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)


NUM_CLASSES = 234
Model loaded successfully from /kaggle/input/models/punyakdei/train-geo-alter2/pytorch/default/1/best_model_vit_fold_0.pth
KNN model loaded from /kaggle/input/models/punyakdei/train-geo-alter2/pytorch/default/1/knn_spatial_fold_0.pkl


In [3]:
test_files = sorted(glob.glob(f"{CFG.SOUNDSCAPE_DIR}/*.ogg"))
if not test_files:
    # Fallback to train soundscapes for local testing
    test_files = sorted(glob.glob(os.path.join(CFG.ROOT_DIR, 'train_soundscapes', '*.ogg')))[:2]

print(f"Found {len(test_files)} soundscape files to process.")

all_predictions = []
all_row_ids     = []

for audio_path in tqdm(test_files, desc="Inference"):
    filename     = os.path.basename(audio_path).replace('.ogg', '')
    data, _      = sf.read(audio_path, dtype='float32')
    if data.ndim > 1:
        data = data.mean(axis=1)

    total_samples        = len(data)
    num_submit_chunks    = math.ceil(total_samples / CFG.CHUNK_LENGTH)
    submit_block_probs   = np.zeros((num_submit_chunks, NUM_CLASSES), dtype=np.float32)

    # ── Step 1: Overlapping 5s window extraction with safe end-padding ──────
    windows, starts = [], []
    for start in range(0, total_samples, CFG.HOP_LENGTH_AUDIO):
        end = start + CFG.CHUNK_LENGTH
        if end > total_samples:
            chunk = data[start:]
            chunk = np.pad(chunk, (0, CFG.CHUNK_LENGTH - len(chunk)))
            windows.append(chunk)
            starts.append(start)
            break
        windows.append(data[start:end])
        starts.append(start)

    if not windows:
        windows.append(np.pad(data, (0, CFG.CHUNK_LENGTH - total_samples)))
        starts.append(0)

    # ── Step 2: Sub-batched GPU inference (VRAM safe) ─────────────────────
    ac_out_list, geo_out_list = [], []

    for i in range(0, len(windows), CFG.SUB_BATCH_SIZE):
        sub_wins      = windows[i : i + CFG.SUB_BATCH_SIZE]
        sub_waveforms = torch.tensor(np.array(sub_wins), dtype=torch.float32).to(device)

        with torch.no_grad():
            sub_mels  = mel_transform(sub_waveforms)
            log_mels  = torch.log(sub_mels + 1e-6)

            n         = len(sub_wins)
            m_min     = log_mels.reshape(n, -1).min(dim=1)[0].reshape(n, 1, 1)
            m_max     = log_mels.reshape(n, -1).max(dim=1)[0].reshape(n, 1, 1)
            log_mels  = (log_mels - m_min) / (m_max - m_min + 1e-6)

            sub_imgs  = log_mels.unsqueeze(1)
            sub_imgs  = F.interpolate(sub_imgs, size=CFG.IMAGE_SIZE,
                                      mode='bilinear', align_corners=False)
            sub_imgs  = sub_imgs.repeat(1, 3, 1, 1)
            sub_imgs  = (sub_imgs - IMG_MEAN) / IMG_STD

            ac_sub, geo_sub = model(sub_imgs)
            ac_out_list.append(torch.sigmoid(ac_sub).cpu().numpy())
            geo_out_list.append(geo_sub.cpu().numpy())

    probs_visual     = np.concatenate(ac_out_list,  axis=0)
    predicted_coords = np.concatenate(geo_out_list, axis=0)

    # ── Step 3: Confidence-gated KNN spatial blending ─────────────────────
    if knn_model is not None:
        probs_knn  = knn_model.predict(predicted_coords)
        max_conf   = np.max(probs_visual, axis=1, keepdims=True)
        high_conf  = max_conf > CFG.CONFIDENCE_THRESHOLD
        final_probs = np.where(
            high_conf,
            probs_visual,
            probs_visual * (1 - CFG.KNN_ALPHA) + probs_knn * CFG.KNN_ALPHA
        )
    else:
        final_probs = probs_visual

    # ── Step 4: Max-pool over overlapping windows per 5s submission block ──
    for i in range(num_submit_chunks):
        block_start = i * CFG.CHUNK_LENGTH
        block_end   = block_start + CFG.CHUNK_LENGTH

        overlapping = []
        for w_idx, w_start in enumerate(starts):
            w_end   = w_start + CFG.CHUNK_LENGTH
            overlap = min(block_end, w_end) - max(block_start, w_start)
            if overlap > CFG.CHUNK_LENGTH * 0.4:
                overlapping.append(final_probs[w_idx])

        if overlapping:
            submit_block_probs[i] = np.max(overlapping, axis=0)

        all_row_ids.append(f"{filename}_{(i + 1) * 5}")
        all_predictions.append(submit_block_probs[i])

    torch.cuda.empty_cache()

# ── Step 5: Build submission, merge against sample_sub to fix row count ───
sub_df = pd.DataFrame(all_predictions, columns=submission_labels)
sub_df.insert(0, 'row_id', all_row_ids)

final_sub = pd.merge(sample_sub[['row_id']], sub_df, on='row_id', how='left').fillna(0.0)
final_sub.to_csv('submission.csv', index=False)
print(f"Done! submission.csv has {len(final_sub)} rows x {len(final_sub.columns)} columns.")


Found 2 soundscape files to process.


Inference:   0%|          | 0/2 [00:00<?, ?it/s]

Done! submission.csv has 3 rows x 235 columns.
